# PROJECT SENTINEL — Notebook 1: Data Pipeline & Feature Engineering
**Layer A** (Snowball DB Engine) + **Layer B** (Feature Intelligence)

Run this notebook to:
- Fetch/update all crypto and macro OHLCV data
- Apply integrity protocol and save to CSVs
- Compute all features (Macro Risk State, BTC/ETH anchors, Altcoin matrices)
- Save feature matrices to `data/working/`

## 0 — Setup

In [ ]:
import os, sys, logging
# Add project root to path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import config
from utils.data_utils import update_all_tickers, load_base_csv
from utils.features import build_all_features, compute_anchor_features, compute_macro_risk_state

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(message)s',
    handlers=[
        logging.FileHandler(os.path.join(config.LOGS_DIR, 'data_pipeline.log')),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)
print('Setup complete.')

In [ ]:
# Create all required directories
for d in [
    config.DATA_BASE,
    os.path.join(config.DATA_BASE, 'crypto'),
    os.path.join(config.DATA_BASE, 'macro'),
    config.DATA_WORKING,
    config.MODELS_DIR,
    config.STATE_DIR,
    config.LOGS_DIR,
]:
    os.makedirs(d, exist_ok=True)
print('Directories ready.')

## Section A — Data Pipeline

In [ ]:
# ── Update ALL tickers (crypto + macro) ──────────────────────────────────────
# Self-healing: each ticker fetches from last saved timestamp.
# If validation fails, it logs and skips (will retry next run).
print('Updating all tickers...')
all_dfs = update_all_tickers()
print(f'Loaded {len(all_dfs)} ticker DataFrames.')

In [ ]:
# ── Data health summary ──────────────────────────────────────────────────────
summary = []
for ticker, df in all_dfs.items():
    if df is None or df.empty:
        summary.append({'ticker': ticker, 'rows': 0, 'from': 'N/A', 'to': 'N/A', 'nulls': 'N/A'})
        continue
    summary.append({
        'ticker': ticker,
        'rows':   len(df),
        'from':   str(df['timestamp'].min())[:16],
        'to':     str(df['timestamp'].max())[:16],
        'nulls':  int(df[['open','high','low','close']].isnull().sum().sum()),
    })
pd.DataFrame(summary).set_index('ticker')

## Section B — Feature Engineering

In [ ]:
# Separate crypto and macro DataFrames
crypto_dfs = {k: v for k, v in all_dfs.items() if '/' in k}
macro_dfs  = {k: v for k, v in all_dfs.items() if '/' not in k}
print(f'Crypto tickers: {list(crypto_dfs.keys())}')
print(f'Macro tickers:  {list(macro_dfs.keys())}')

In [ ]:
# ── Model A: Macro Risk State ────────────────────────────────────────────────
print('Computing Macro_Risk_State...')
macro_state = compute_macro_risk_state(macro_dfs)
print(f'Macro_Risk_State: {len(macro_state)} rows | range [{macro_state.min():.3f}, {macro_state.max():.3f}]')

fig, ax = plt.subplots(figsize=(14, 3))
macro_state.plot(ax=ax, color='steelblue')
ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax.set_title('Macro Risk State (−1 = risk-off, +1 = risk-on)')
ax.set_ylabel('Score')
plt.tight_layout()
plt.show()

In [ ]:
# ── Model B: BTC & ETH Anchor Features ──────────────────────────────────────
print('Computing BTC anchor features...')
btc_anchor = compute_anchor_features(crypto_dfs['BTC/USDT'], 'BTC')
print(f'BTC anchor: {btc_anchor.shape} | columns: {list(btc_anchor.columns)}')

print('Computing ETH anchor features...')
eth_anchor = compute_anchor_features(crypto_dfs['ETH/USDT'], 'ETH')
print(f'ETH anchor: {eth_anchor.shape}')

In [ ]:
# Visualise key BTC anchor features
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
for ax, col, title in zip(axes,
    ['BTC_Regime', 'BTC_Vol_State', 'BTC_Breakout_Prob', 'BTC_Direction_Bias'],
    ['Regime (+1=trend, −1=range)', 'Volatility State (+1=expansion)',
     'Breakout Probability', 'Directional Bias']):
    if col in btc_anchor.columns:
        btc_anchor.set_index('timestamp')[col].tail(500).plot(ax=ax)
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.5)
    ax.set_title(f'BTC {title}')
plt.tight_layout()
plt.show()

In [ ]:
# ── Model C: Altcoin Feature Matrices ────────────────────────────────────────
print('Building all altcoin feature matrices...')
feature_dfs = build_all_features(crypto_dfs, macro_dfs)
print(f'Built features for: {list(feature_dfs.keys())}')

In [ ]:
# ── NaN audit ────────────────────────────────────────────────────────────────
audit = []
for ticker, feat in feature_dfs.items():
    total_cells = feat.shape[0] * feat.shape[1]
    null_cells  = feat.isnull().sum().sum()
    inf_cells   = np.isinf(feat.select_dtypes('number')).sum().sum()
    audit.append({
        'ticker':  ticker,
        'rows':    feat.shape[0],
        'cols':    feat.shape[1],
        'nulls':   int(null_cells),
        'infs':    int(inf_cells),
        'null_%':  f'{null_cells/total_cells*100:.1f}%',
    })
pd.DataFrame(audit).set_index('ticker')

In [ ]:
# ── Save feature matrices to data/working/ ────────────────────────────────────
for ticker, feat in feature_dfs.items():
    safe = ticker.replace('/', '_')
    path = os.path.join(config.DATA_WORKING, f'{safe}_features.parquet')
    feat.to_parquet(path, index=False)
    print(f'Saved {path}  ({feat.shape[0]} rows × {feat.shape[1]} cols)')
print('All feature matrices saved.')

In [ ]:
# ── Feature columns preview (one coin) ───────────────────────────────────────
sample_ticker = list(feature_dfs.keys())[0]
sample_feat   = feature_dfs[sample_ticker]
print(f'Feature columns for {sample_ticker} ({len(sample_feat.columns)} total):')
print(list(sample_feat.columns))
sample_feat.tail(3)